In [1]:
import polars as pl
import awpy
from xgboost import XGBClassifier
import sys
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
sys.path.append(str(Path.cwd().parent))
from src.wpa_engine import calculate_match_wpa, aggregate_player_wpa

In [2]:
train_df = pl.read_parquet("../data/processed_demos/tick_dataset.parquet")
econ_df = pl.read_parquet("../data/processed_demos/economy_dataset.parquet")
features = ["ct_alive", "t_alive", "ct_equip_value", "t_equip_value", "equip_value_ratio", "seconds_elapsed", "is_bomb_planted", "seconds_since_plant"]
econ_features = ["ct_equip_value", "t_equip_value", "equip_value_ratio"]


In [3]:
econ_df

round_num,ct_equip_value,ct_buy_type,t_equip_value,t_buy_type,ct_win,equip_value_diff,equip_value_ratio,match_id
i32,i32,str,i32,str,i32,i32,f64,str
1,4800,"""pistol""",4200,"""pistol""",1,600,1.142823,"""aurora-vs-vitality-m1-anubis"""
2,22650,"""full""",14650,"""force""",1,8000,1.546038,"""aurora-vs-vitality-m1-anubis"""
3,28100,"""full""",2100,"""eco""",1,26000,13.375059,"""aurora-vs-vitality-m1-anubis"""
4,29500,"""full""",24050,"""full""",0,5450,1.226602,"""aurora-vs-vitality-m1-anubis"""
5,28100,"""full""",27650,"""full""",0,450,1.016274,"""aurora-vs-vitality-m1-anubis"""
…,…,…,…,…,…,…,…,…
24,31050,"""full""",18100,"""force""",1,12950,1.71543,"""vitality-vs-mouz-m3-dust2"""
25,31600,"""full""",27350,"""full""",1,4250,1.155387,"""vitality-vs-mouz-m3-dust2"""
26,32150,"""full""",27350,"""full""",1,4800,1.175496,"""vitality-vs-mouz-m3-dust2"""


In [4]:
X_train = train_df[features].to_numpy()
Y_train = train_df["ct_win"].to_numpy().flatten()
#scaler = StandardScaler()
#X_train_scaled = scaler.fit_transform(X_train)

In [5]:
xgb = XGBClassifier(n_estimators=150, max_depth = 8, learning_rate = 0.1, objective="binary:logistic")
xgb.fit(X_train, Y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [6]:
X_econ_train = econ_df[econ_features].to_numpy()
Y_econ_train = econ_df["ct_win"].to_numpy().flatten()
#scaler_econ = StandardScaler()
#X_econ_train_scaled = scaler_econ.fit_transform(X_econ_train)

In [7]:
econ_lr = LogisticRegression(random_state = 42)
econ_lr = econ_lr.fit(X_econ_train, Y_econ_train)

In [ ]:
DEMO_PATH = "../data/raw_demos/spirit-vs-furia-m1-ancient.dem"
dem = awpy.Demo(DEMO_PATH)

In [11]:
wpa_events = calculate_match_wpa(dem, xgb, econ_lr)
leaderboard = aggregate_player_wpa(wpa_events)

['round_num', 'tick', 'event_type', 'ct_death', 't_death', 'freeze_end_tick', 'ct_win', 'ct_equip_value', 't_equip_value', 'equip_value_ratio', 'seconds_elapsed', 'ct_alive', 't_alive', 'is_plant_event', 'is_bomb_planted', 'plant_tick', 'seconds_since_plant', 'ct_win_prob']
['round_num', 'ct_equip_value', 'ct_buy_type', 't_equip_value', 't_buy_type', 'ct_win', 'equip_value_diff', 'equip_value_ratio', 'round_start_prob']
shape: (224, 25)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ rou ┆ tic ┆ eve ┆ ct_ ┆ t_d ┆ fre ┆ ct_ ┆ ct_ ┆ t_e ┆ equ ┆ sec ┆ ct_ ┆ t_a ┆ is_ ┆ is_ ┆ pla ┆ sec ┆ ct_ ┆ rou ┆ pre ┆ del ┆ act ┆ act ┆ vic ┆ vic │
│ nd_ ┆ k   ┆ nt_ ┆ dea ┆ eat ┆ eze ┆ win ┆ equ ┆ qui ┆ ip_ ┆ ond ┆ ali ┆ liv ┆ pla ┆ bom ┆ nt_ ┆ ond ┆ win ┆ nd_ ┆ v_c ┆ ta_ ┆ or_ ┆ or_ ┆ tim ┆ tim │
│ num ┆ --- ┆ typ ┆ th  ┆ h   ┆ _en ┆ --- ┆ ip_ ┆ p_v ┆ val ┆ s_e ┆ ve  ┆ e   ┆ nt_ ┆ b_p ┆ tic ┆ s_s ┆

In [10]:
print("=== Match WPA Player Leaderboard ===")
print(leaderboard)

=== Match WPA Player Leaderboard ===
shape: (10, 7)
┌─────────────┬──────────────┬─────────┬───────────────┬──────────────┬──────────────┬─────────────┐
│ player_name ┆ positive_wpa ┆ actions ┆ player_name_r ┆ negative_wpa ┆ actions_righ ┆ average_wpa │
│ ---         ┆ ---          ┆ ---     ┆ ight          ┆ ---          ┆ t            ┆ ---         │
│ str         ┆ f64          ┆ f64     ┆ ---           ┆ f64          ┆ ---          ┆ f64         │
│             ┆              ┆         ┆ str           ┆              ┆ f64          ┆             │
╞═════════════╪══════════════╪═════════╪═══════════════╪══════════════╪══════════════╪═════════════╡
│ mezii       ┆ 1.955567     ┆ 33.0    ┆ mezii         ┆ -0.657257    ┆ 19.0         ┆ 0.076371    │
│ Jimpphat    ┆ 0.793154     ┆ 21.0    ┆ Jimpphat      ┆ -0.048647    ┆ 23.0         ┆ 0.043795    │
│ ropz        ┆ 0.850236     ┆ 25.0    ┆ ropz          ┆ -0.216463    ┆ 17.0         ┆ 0.037281    │
│ flameZ      ┆ 0.193237     ┆ 19.0    